In [10]:
import os

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from behave_analysis.analyze.filtering_data.filtering_functions import filter_video_dataframe

In [11]:
# JAL1 SEQ3
JAL1_SEQ3_video_df = r"Z:\Jasmine_Laurence\Experimental_Data\JAL001\001_seq1_3_2023_03_17T08_38_03\processed_data\full_video_dataframe.csv"
seq3_video_df = pl.read_csv(JAL1_SEQ3_video_df)

# JAL1 SEQ2
JAL1_SEQ2_video_df = r"Z:\Jasmine_Laurence\Experimental_Data\JAL001\001_seq1_2_2023_03_14T08_11_32\processed_data\full_video_dataframe.csv" 
seq2_video_df = pl.read_csv(JAL1_SEQ2_video_df)

# JAL2 SEQ2
JAL2_SEQ2_video_df =  r"Z:\Jasmine_Laurence\Experimental_Data\JAL002\002_seq1_2_2023_04_25T07_32_30\processed_data\full_video_dataframe.csv"
jal2_seq2_df = pl.read_csv(JAL2_SEQ2_video_df)

# JAl2 SEQ3
JAL2_SEQ3_video_df =  r"Z:\Jasmine_Laurence\Experimental_Data\JAL002\002_Sequence1_3_2023_04_28T08_31_30\processed_data\full_video_dataframe.csv"
jal2_seq3_df = pl.read_csv(JAL2_SEQ3_video_df)

In [12]:
jal2_seq3_df.columns

['frames',
 'hdir',
 'mouse_x_position',
 'mouse_y_position',
 'speed',
 'OutofshelterIdx',
 'EscapePeriod',
 'shelter',
 'barrier_present',
 'barrier_flipped',
 'hsa',
 'h_preflipbar_a',
 'h_postflipbar_a',
 'h_bar_centre_a',
 'head_randP_0',
 'head_randP_1',
 'head_randP_2',
 'head_randP_3',
 'head_randP_4',
 'head_randP_5',
 'head_randP_6',
 'head_randP_7',
 'head_randP_8',
 'head_randP_9',
 'head_randP_10',
 'head_randP_11',
 'head_randP_12',
 'head_randP_13',
 'head_randP_14',
 'head_randP_15',
 'head_randP_16',
 'head_randP_17',
 'head_randP_18',
 'head_randP_19',
 'head_randP_20',
 'head_randP_21',
 'head_randP_22',
 'head_randP_23',
 'head_randP_24',
 'head_randP_25',
 'head_randP_26',
 'head_randP_27',
 'head_randP_28',
 'head_randP_29',
 'head_randP_30',
 'head_randP_31',
 'head_randP_32',
 'head_randP_33',
 'head_randP_34',
 'head_randP_35',
 'head_randP_36',
 'head_randP_37',
 'head_randP_38',
 'head_randP_39',
 'head_randP_40',
 'head_randP_41',
 'head_randP_42',
 'head_ra

In [13]:


# Arena parameters
arena_radius = 460  # pixels (1 pixel = 1 cm)
arena_size = 1024  # size of the arena (1024x1024 pixels)

sessions = {
    "JAL1_SEQ3": seq3_video_df,
    "JAL1_SEQ2": seq2_video_df,
    "JAL2_SEQ2": jal2_seq2_df,
    "JAL2_SEQ3": jal2_seq3_df,
}

def quantify_arena_coverage(video_df, arena_radius, arena_size, bins=100):
    """Quantify gaps in arena coverage for a single session."""
    # Extract mouse positions
    x_positions = video_df["mouse_x_position"].to_numpy()
    y_positions = video_df["mouse_y_position"].to_numpy()

    # Create a 2D histogram of visited positions with larger bins
    hist, x_edges, y_edges = np.histogram2d(
        x_positions, y_positions, bins=bins, range=[[0, arena_size], [0, arena_size]]
    )

    # Create a mask for the circular arena
    y, x = np.ogrid[:bins, :bins]
    center = bins // 2
    mask = (x - center)**2 + (y - center)**2 <= (arena_radius / (arena_size / bins))**2

    # Calculate visited pixels
    visited = hist > 0
    visited_area = np.sum(visited & mask)

    # Calculate total area of the arena
    total_area = np.sum(mask)

    # Calculate unvisited area
    unvisited_area = total_area - visited_area

    # Divide the arena into quadrants and calculate unvisited area for each
    quadrants = {
        "top-left": mask & (x < center) & (y < center),
        "top-right": mask & (x >= center) & (y < center),
        "bottom-left": mask & (x < center) & (y >= center),
        "bottom-right": mask & (x >= center) & (y >= center),
    }
    quadrant_unvisited = {q: np.sum((~visited) & quadrants[q]) for q in quadrants}

    # Calculate percentages
    unvisited_percentage = (unvisited_area / total_area) * 100
    quadrant_percentages = {q: (quadrant_unvisited[q] / np.sum(quadrants[q])) * 100 for q in quadrants}

    return unvisited_percentage, quadrant_percentages

# Quantify coverage for all sessions
results = {}
for session_name, video_df in sessions.items():
    unvisited_percentage, quadrant_percentages = quantify_arena_coverage(video_df, arena_radius, arena_size, bins=100)
    results[session_name] = {
        "unvisited_percentage": unvisited_percentage,
        "quadrant_percentages": quadrant_percentages,
    }

# Aggregate quadrant percentages across sessions
quadrants = ["top-left", "top-right", "bottom-left", "bottom-right"]
quadrant_data = {q: [] for q in quadrants}

for session_name, data in results.items():
    for q in quadrants:
        quadrant_data[q].append(data["quadrant_percentages"][q])

# Calculate mean and SEM for each quadrant
means = [np.mean(quadrant_data[q]) for q in quadrants]
sems = [np.std(quadrant_data[q], ddof=1) / np.sqrt(len(quadrant_data[q])) for q in quadrants]

# Plot aggregated data
plt.rcParams.update({
    "font.size": 18,
    "font.family": "Arial",
    "axes.labelsize": 20,
    "axes.titlesize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
})

fig, ax = plt.subplots(figsize=(8, 6))

x = np.arange(len(quadrants))
bar_width = 0.6

# Plot bars with error bars
ax.bar(x, means, yerr=sems, capsize=5, color="lightgrey", edgecolor="black", linewidth=1.5, width=bar_width)

# Add crosses for individual session data points
for i, q in enumerate(quadrants):
    for val in quadrant_data[q]:
        ax.plot(i, val, marker='x', color='black', markersize=10, markeredgewidth=1.5)

# Add labels and formatting
ax.set_xticks(x)
ax.set_xticklabels(quadrants)
ax.set_ylabel("Unvisited Area (%)")
ax.set_xlabel("Quadrants")
ax.set_title("Unvisited Arena Coverage by Quadrant")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)
ax.tick_params(width=1.5, length=6)
plt.tight_layout()
plt.show()
save_path = r"Z:\Laurence\thesis\figures\two_edge_paradigm"
fig.savefig(os.path.join(save_path, "two_edge_quandrant_coverage.eps"), format='eps', bbox_inches='tight', dpi=300)

In [14]:
import matplotlib.pyplot as plt
import numpy as np

# Arena parameters
arena_radius = 460  # pixels (1 pixel = 1 cm)

# Create the figure and axis
fig, ax = plt.subplots(figsize=(6, 6))

# Draw the circular arena
circle = plt.Circle((0, 0), arena_radius, color="lightgrey", alpha=0.5, edgecolor="black", linewidth=1.5)
ax.add_artist(circle)

# Draw quadrant dividing lines
ax.plot([-arena_radius, arena_radius], [0, 0], color="black", linewidth=1.5)  # Horizontal line
ax.plot([0, 0], [-arena_radius, arena_radius], color="black", linewidth=1.5)  # Vertical line

# Label quadrants
ax.text(-arena_radius / 2, arena_radius / 2, "Top-Left", fontsize=16, ha="center", va="center", fontweight="bold")
ax.text(arena_radius / 2, arena_radius / 2, "Top-Right", fontsize=16, ha="center", va="center", fontweight="bold")
ax.text(-arena_radius / 2, -arena_radius / 2, "Bottom-Left", fontsize=16, ha="center", va="center", fontweight="bold")
ax.text(arena_radius / 2, -arena_radius / 2, "Bottom-Right", fontsize=16, ha="center", va="center", fontweight="bold")

# Set axis limits and aspect ratio
ax.set_xlim(-arena_radius - 50, arena_radius + 50)
ax.set_ylim(-arena_radius - 50, arena_radius + 50)
ax.set_aspect("equal", adjustable="datalim")

# Remove axis ticks and labels
ax.axis("off")

# Add title
ax.set_title("Arena Quadrants", fontsize=20, fontweight="bold", pad=20)

plt.tight_layout()
plt.show()
fig.savefig(os.path.join(save_path, "quandrant_explainer.eps"), format='eps', bbox_inches='tight', dpi=300)

C:\Users\laurence\AppData\Local\Temp\ipykernel_23496\1425720709.py:11: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  circle = plt.Circle((0, 0), arena_radius, color="lightgrey", alpha=0.5, edgecolor="black", linewidth=1.5)
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [15]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

In [16]:
experiments_objects = [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

In [17]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from behave_analysis.process.session import get_experiment

# Arena parameters
arena_radius = 460  # pixels (1 pixel = 1 cm)
arena_size = 1024  # size of the arena (1024x1024 pixels)
bins = 100  # Number of bins for histogram

# Two-edge sessions
two_edge_sessions = {
    "JAL1_SEQ3": seq3_video_df,
    "JAL1_SEQ2": seq2_video_df,
    "JAL2_SEQ2": jal2_seq2_df,
    "JAL2_SEQ3": jal2_seq3_df,
}

# Function to quantify arena coverage
def quantify_arena_coverage(video_df, arena_radius, arena_size, bins=100):
    """Quantify gaps in arena coverage for a single session."""
    # Extract mouse positions
    x_positions = video_df["mouse_x_position"].to_numpy()
    y_positions = video_df["mouse_y_position"].to_numpy()

    # Create a 2D histogram of visited positions
    hist, x_edges, y_edges = np.histogram2d(
        x_positions, y_positions, bins=bins, range=[[0, arena_size], [0, arena_size]]
    )

    # Create a mask for the circular arena
    y, x = np.ogrid[:bins, :bins]
    center = bins // 2
    mask = (x - center)**2 + (y - center)**2 <= (arena_radius / (arena_size / bins))**2

    # Calculate visited pixels
    visited = hist > 0
    visited_area = np.sum(visited & mask)

    # Calculate total area of the arena
    total_area = np.sum(mask)

    # Divide the arena into quadrants and calculate unvisited area for each
    quadrants = {
        "top-left": mask & (x < center) & (y < center),
        "top-right": mask & (x >= center) & (y < center),
        "bottom-left": mask & (x < center) & (y >= center),
        "bottom-right": mask & (x >= center) & (y >= center),
    }
    quadrant_unvisited = {q: np.sum((~visited) & quadrants[q]) for q in quadrants}

    # Calculate percentages
    quadrant_percentages = {q: (quadrant_unvisited[q] / np.sum(quadrants[q])) * 100 for q in quadrants}

    return quadrant_percentages

# Calculate quadrant percentages for two-edge sessions
two_edge_quadrant_data = {q: [] for q in ["top-left", "top-right", "bottom-left", "bottom-right"]}
for session_name, video_df in two_edge_sessions.items():
    quadrant_percentages = quantify_arena_coverage(video_df, arena_radius, arena_size, bins)
    for q in quadrant_percentages:
        two_edge_quadrant_data[q].append(quadrant_percentages[q])

# Calculate quadrant percentages for one-edge task (experimental objects)
one_edge_quadrant_data = {q: [] for q in ["top-left", "top-right", "bottom-left", "bottom-right"]}
for session in experiments_objects:
    loaded_session = get_experiment(session)
    video_df_path = os.path.join(
        loaded_session.base_path,
        loaded_session.processed_path,
        "full_video_dataframe.csv"
    )
    if os.path.exists(video_df_path):
        video_df = pl.read_csv(video_df_path)
        quadrant_percentages = quantify_arena_coverage(video_df, arena_radius, arena_size, bins)
        for q in quadrant_percentages:
            one_edge_quadrant_data[q].append(quadrant_percentages[q])

# Calculate mean and SEM for each quadrant for both groups
quadrants = ["top-left", "top-right", "bottom-left", "bottom-right"]
two_edge_means = [np.mean(two_edge_quadrant_data[q]) for q in quadrants]
two_edge_sems = [np.std(two_edge_quadrant_data[q], ddof=1) / np.sqrt(len(two_edge_quadrant_data[q])) for q in quadrants]
one_edge_means = [np.mean(one_edge_quadrant_data[q]) for q in quadrants]
one_edge_sems = [np.std(one_edge_quadrant_data[q], ddof=1) / np.sqrt(len(one_edge_quadrant_data[q])) for q in quadrants]

# Plot comparison
plt.rcParams.update({
    "font.size": 18,
    "font.family": "Arial",
    "axes.labelsize": 20,
    "axes.titlesize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
})

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(quadrants))
bar_width = 0.35

# Plot bars for two-edge sessions
ax.bar(
    x - bar_width/2, two_edge_means, yerr=two_edge_sems, width=bar_width,
    color="dimgrey", label="Two-Edge Sessions", capsize=5, edgecolor='black', linewidth=1.5
)

# Plot bars for one-edge task
ax.bar(
    x + bar_width/2, one_edge_means, yerr=one_edge_sems, width=bar_width,
    color="lightgrey", label="One-Edge Task", capsize=5, edgecolor='black', linewidth=1.5
)

# Add labels and formatting
ax.set_xticks(x)
ax.set_xticklabels(quadrants)
ax.set_ylabel("Unvisited Area (%)")
ax.set_xlabel("Quadrants")
ax.set_title("Comparison of Unvisited Arena Coverage by Quadrant")
ax.legend(facecolor="white", frameon=False, loc='upper left')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)
ax.tick_params(width=1.5, length=6)
plt.tight_layout()

# Save figure as EPS for publication
save_path = r"Z:\Laurence\thesis\figures\two_edge_paradigm"
fig.savefig(os.path.join(save_path, "quadrant_coverage_comparison.eps"), format='eps', bbox_inches='tight', dpi=300)
plt.show()

KeyboardInterrupt: 